In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("EchoChain") \
    .getOrCreate()

print("PySpark started successfully!")

PySpark started successfully!


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving EchoChain_Circular_Economy_Dataset_10000.csv to EchoChain_Circular_Economy_Dataset_10000.csv


In [ ]:
  import os

print(os.listdir("/content"))

['.config', 'EchoChain_Circular_Economy_Dataset_10000.csv', 'EchoChain_Circular_Economy_Analytics.ipynb', 'EchoChain_Circular_Economy_Dataset_10000 (1).csv', 'sample_data']


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("EchoChain") \
    .getOrCreate()

df = spark.read.csv(
    "/content/EchoChain_Circular_Economy_Dataset_10000.csv",
    header=True,
    inferSchema=True
)

df.show(5)

+----------+----------------+---------------+------------+------------------+------------------+-------------+------------------+---------------+------------+-----------+---------------+-----------------+--------------------+-----------------+--------------------+----------------------+----------------+------------------+-----------+--------------+------------------+---------------+---------------+----------------+--------------------+-----------------+
|Product_ID|Product_Category|   Product_Type|Manufacturer|Manufacturing_Date|Original_Price_INR|Material_Type|Material_Weight_KG|Customer_Region|Usage_Months|Return_Date|  Return_Reason|Product_Condition|  Collection_Channel|Inspection_Result|Refurbishment_Status|Refurbishment_Cost_INR|Resale_Price_INR|    Resale_Channel|Resale_Date|Recycle_Status|Recycled_Weight_KG|Waste_Weight_KG|Carbon_Saved_KG|Lifecycle_Status|Circularity_Rate_Pct|Profit_Margin_INR|
+----------+----------------+---------------+------------+------------------+-------

In [ ]:
df.printSchema()

root
 |-- Product_ID: string (nullable = true)
 |-- Product_Category: string (nullable = true)
 |-- Product_Type: string (nullable = true)
 |-- Manufacturer: string (nullable = true)
 |-- Manufacturing_Date: date (nullable = true)
 |-- Original_Price_INR: double (nullable = true)
 |-- Material_Type: string (nullable = true)
 |-- Material_Weight_KG: double (nullable = true)
 |-- Customer_Region: string (nullable = true)
 |-- Usage_Months: integer (nullable = true)
 |-- Return_Date: date (nullable = true)
 |-- Return_Reason: string (nullable = true)
 |-- Product_Condition: string (nullable = true)
 |-- Collection_Channel: string (nullable = true)
 |-- Inspection_Result: string (nullable = true)
 |-- Refurbishment_Status: string (nullable = true)
 |-- Refurbishment_Cost_INR: double (nullable = true)
 |-- Resale_Price_INR: double (nullable = true)
 |-- Resale_Channel: string (nullable = true)
 |-- Resale_Date: date (nullable = true)
 |-- Recycle_Status: string (nullable = true)
 |-- Recycl

In [ ]:
print("Total Records:", df.count())

Total Records: 10000


In [ ]:
print("Total Columns:", len(df.columns))

Total Columns: 27


In [ ]:
df.groupBy("Product_Category") \
  .count() \
  .orderBy("count", ascending=False) \
  .show()

+--------------------+-----+
|    Product_Category|count|
+--------------------+-----+
|         Electronics| 2564|
|Industrial Equipment| 2536|
|          Automotive| 2454|
|      Home Appliance| 2446|
+--------------------+-----+



In [ ]:
df.groupBy("Product_Condition") \
  .count() \
  .show()

+-----------------+-----+
|Product_Condition|count|
+-----------------+-----+
|        Excellent| 1551|
|             Good| 4497|
|             Fair| 2982|
|             Poor|  970|
+-----------------+-----+



In [ ]:
df.groupBy("Product_Condition") \
  .count() \
  .show()

+-----------------+-----+
|Product_Condition|count|
+-----------------+-----+
|        Excellent| 1551|
|             Good| 4497|
|             Fair| 2982|
|             Poor|  970|
+-----------------+-----+



In [ ]:

from pyspark.sql.functions import avg, round

df.select(
    round(avg("Circularity_Rate_Pct"), 2).alias("Average_Circularity_Rate")
).show()

+------------------------+
|Average_Circularity_Rate|
+------------------------+
|                   46.94|
+------------------------+



In [ ]:
from pyspark.sql.functions import count, when

recycling_rate = df.select(
    round(
        (count(when(df["Recycle_Status"] == "Recycled", True)) / count("*")) * 100,
        2
    ).alias("Recycling_Rate_Pct")
)

recycling_rate.show()

+------------------+
|Recycling_Rate_Pct|
+------------------+
|             29.74|
+------------------+



In [ ]:
from pyspark.sql.functions import sum

df.select(
    round(sum("Recycled_Weight_KG"), 2).alias("Total_Recycled_Weight_KG")
).show()

+------------------------+
|Total_Recycled_Weight_KG|
+------------------------+
|                66266.46|
+------------------------+



In [ ]:
df.groupBy("Refurbishment_Status") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()

+--------------------+-----+
|Refurbishment_Status|count|
+--------------------+-----+
|         Refurbished| 6831|
|     Not Refurbished| 3169|
+--------------------+-----+



In [ ]:
df.groupBy(
    "Product_Category",
    "Refurbishment_Status"
).count().orderBy(
    "Product_Category",
    "count",
    ascending=False
).show()

+--------------------+--------------------+-----+
|    Product_Category|Refurbishment_Status|count|
+--------------------+--------------------+-----+
|Industrial Equipment|         Refurbished| 1673|
|Industrial Equipment|     Not Refurbished|  863|
|      Home Appliance|         Refurbished| 1701|
|      Home Appliance|     Not Refurbished|  745|
|         Electronics|         Refurbished| 1787|
|         Electronics|     Not Refurbished|  777|
|          Automotive|         Refurbished| 1670|
|          Automotive|     Not Refurbished|  784|
+--------------------+--------------------+-----+



In [ ]:
df.groupBy("Product_Category") \
    .agg(
        round(avg("Refurbishment_Cost_INR"), 2)
        .alias("Average_Refurbishment_Cost_INR")
    ) \
    .orderBy("Average_Refurbishment_Cost_INR", ascending=False) \
    .show()

+--------------------+------------------------------+
|    Product_Category|Average_Refurbishment_Cost_INR|
+--------------------+------------------------------+
|          Automotive|                       8190.32|
|         Electronics|                        6733.5|
|Industrial Equipment|                       6262.88|
|      Home Appliance|                       5754.18|
+--------------------+------------------------------+



In [ ]:
df.select(
    round(sum("Resale_Price_INR"), 2)
    .alias("Total_Resale_Revenue_INR")
).show()

+------------------------+
|Total_Resale_Revenue_INR|
+------------------------+
|          4.7665933664E8|
+------------------------+



In [ ]:
df.groupBy("Product_Category") \
    .agg(
        round(sum("Resale_Price_INR"), 2)
        .alias("Total_Resale_Revenue_INR")
    ) \
    .orderBy("Total_Resale_Revenue_INR", ascending=False) \
    .show()

+--------------------+------------------------+
|    Product_Category|Total_Resale_Revenue_INR|
+--------------------+------------------------+
|          Automotive|          1.4230178072E8|
|         Electronics|          1.1851656357E8|
|Industrial Equipment|          1.1782354816E8|
|      Home Appliance|           9.801744419E7|
+--------------------+------------------------+



In [ ]:
df.groupBy("Product_Category") \
    .agg(
        round(avg("Resale_Price_INR"), 2)
        .alias("Average_Resale_Price_INR")
    ) \
    .orderBy("Average_Resale_Price_INR", ascending=False) \
    .show()

+--------------------+------------------------+
|    Product_Category|Average_Resale_Price_INR|
+--------------------+------------------------+
|          Automotive|                57987.69|
|Industrial Equipment|                46460.39|
|         Electronics|                46223.31|
|      Home Appliance|                40072.54|
+--------------------+------------------------+



In [ ]:
df.groupBy("Lifecycle_Status") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()

+----------------+-----+
|Lifecycle_Status|count|
+----------------+-----+
|          Resold| 9030|
|        Recycled|  970|
+----------------+-----+



In [ ]:
df.groupBy("Return_Reason") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()

+---------------+-----+
|  Return_Reason|count|
+---------------+-----+
|       Trade-In| 1747|
|Warranty Return| 1677|
|        Upgrade| 1670|
|         Defect| 1660|
|     End of Use| 1656|
|Customer Return| 1590|
+---------------+-----+



In [ ]:
df.groupBy("Product_Condition") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()

+-----------------+-----+
|Product_Condition|count|
+-----------------+-----+
|             Good| 4497|
|             Fair| 2982|
|        Excellent| 1551|
|             Poor|  970|
+-----------------+-----+



In [ ]:
df.select(
    round(sum("Carbon_Saved_KG"), 2)
    .alias("Total_Carbon_Saved_KG")
).show()

+---------------------+
|Total_Carbon_Saved_KG|
+---------------------+
|            531344.28|
+---------------------+



In [ ]:
df.groupBy("Product_Category") \
    .agg(
        round(sum("Carbon_Saved_KG"), 2)
        .alias("Carbon_Saved_KG")
    ) \
    .orderBy("Carbon_Saved_KG", ascending=False) \
    .show()

+--------------------+---------------+
|    Product_Category|Carbon_Saved_KG|
+--------------------+---------------+
|      Home Appliance|      203462.33|
|          Automotive|      169903.08|
|Industrial Equipment|      138666.23|
|         Electronics|       19312.64|
+--------------------+---------------+



In [ ]:
df.select(
    round(sum("Waste_Weight_KG"), 2)
    .alias("Total_Waste_KG")
).show()

+--------------+
|Total_Waste_KG|
+--------------+
|      194967.6|
+--------------+



In [ ]:
df.groupBy("Customer_Region") \
    .agg(
        round(avg("Circularity_Rate_Pct"), 2)
        .alias("Average_Circularity_Rate"),
        round(sum("Carbon_Saved_KG"), 2)
        .alias("Carbon_Saved_KG"),
        round(sum("Recycled_Weight_KG"), 2)
        .alias("Recycled_Weight_KG")
    ) \
    .orderBy("Average_Circularity_Rate", ascending=False) \
    .show()

+---------------+------------------------+---------------+------------------+
|Customer_Region|Average_Circularity_Rate|Carbon_Saved_KG|Recycled_Weight_KG|
+---------------+------------------------+---------------+------------------+
|         Mumbai|                   47.72|       65930.94|           8424.99|
|     Coimbatore|                   47.71|       65839.63|           8410.76|
|          Delhi|                   47.49|       67174.69|           8670.95|
|      Hyderabad|                   47.35|       65494.15|           8297.19|
|        Chennai|                   46.77|        69973.7|           8415.37|
|      Bengaluru|                   46.51|       67398.02|           8412.53|
|           Pune|                   46.33|       67681.44|           8409.44|
|        Kolkata|                   45.74|       61851.71|           7225.23|
+---------------+------------------------+---------------+------------------+



In [ ]:
final_df = df.select(
    "Product_ID",
    "Product_Category",
    "Product_Type",
    "Manufacturer",
    "Customer_Region",
    "Original_Price_INR",
    "Product_Condition",
    "Usage_Months",
    "Return_Reason",
    "Refurbishment_Status",
    "Refurbishment_Cost_INR",
    "Resale_Price_INR",
    "Resale_Channel",
    "Recycle_Status",
    "Recycled_Weight_KG",
    "Waste_Weight_KG",
    "Carbon_Saved_KG",
    "Lifecycle_Status",
    "Circularity_Rate_Pct",
    "Profit_Margin_INR"
)

final_df.show(5)

+----------+----------------+---------------+------------+---------------+------------------+-----------------+------------+---------------+--------------------+----------------------+----------------+------------------+--------------+------------------+---------------+---------------+----------------+--------------------+-----------------+
|Product_ID|Product_Category|   Product_Type|Manufacturer|Customer_Region|Original_Price_INR|Product_Condition|Usage_Months|  Return_Reason|Refurbishment_Status|Refurbishment_Cost_INR|Resale_Price_INR|    Resale_Channel|Recycle_Status|Recycled_Weight_KG|Waste_Weight_KG|Carbon_Saved_KG|Lifecycle_Status|Circularity_Rate_Pct|Profit_Margin_INR|
+----------+----------------+---------------+------------+---------------+------------------+-----------------+------------+---------------+--------------------+----------------------+----------------+------------------+--------------+------------------+---------------+---------------+----------------+-----------

In [ ]:
final_df.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("/content/EchoChain_PowerBI_Data")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')